In [1]:
import numpy as np
from clifford.g4 import layout
from pyganja import GanjaScene, draw

e1, e2, e3, e4 = layout.basis_vectors.values()

# Scaling factor for the whole geometry
geom_scale = 0.5          # <-- make this smaller to shrink the objects

# Parameters for grid resolution
n_theta = 8
n_phi = 12
theta1_vals = np.linspace(0.01, np.pi - 0.01, n_theta)
theta2_vals = np.linspace(0.01, np.pi - 0.01, n_theta)
phi_vals = np.linspace(0, 2*np.pi, n_phi, endpoint=False)

vertices = []
rotors = []
idx_map = {}

# Generate grid points and rotors
for i, t1 in enumerate(theta1_vals):
    for j, t2 in enumerate(theta2_vals):
        for k, phi in enumerate(phi_vals):
            x1 = np.cos(t1)
            x2 = np.sin(t1) * np.cos(t2)
            x3 = np.sin(t1) * np.sin(t2) * np.cos(phi)
            x4 = np.sin(t1) * np.sin(t2) * np.sin(phi)
            # Apply scaling to the vector
            v = geom_scale * (x1 * e1 + x2 * e2 + x3 * e3 + x4 * e4)
            R = (1 + v * e1) / abs(1 + v * e1)   # rotor still based on scaled v
            vertices.append(v)
            rotors.append(R)
            idx_map[(i, j, k)] = len(vertices) - 1

# Build edges
edges = []
for i in range(n_theta):
    for j in range(n_theta):
        for k in range(n_phi):
            vi = idx_map[(i, j, k)]
            if i + 1 < n_theta:
                edges.append((vertices[vi], vertices[idx_map[(i+1, j, k)]]))
            if j + 1 < n_theta:
                edges.append((vertices[vi], vertices[idx_map[(i, j+1, k)]]))
            k_next = (k + 1) % n_phi
            edges.append((vertices[vi], vertices[idx_map[(i, j, k_next)]]))

# ---------- Scene 1: Hypersphere surface ----------
scene = GanjaScene()
for v in vertices:
    scene.add_object(v, color=(0, 200, 255))
for v1, v2 in edges:
    scene.add_object(v1 ^ v2, color=(150, 200, 255, 77))

draw(scene, sig=layout.sig, scale=1.0,          # keep scale=1 or omit
     point_size=0.01, new_window=True)          # smaller points as well

# ---------- Scene 2: Rotor manifold ----------
# Get indices for bivectors
idx12 = layout.bladeTupList.index((1, 2))
idx13 = layout.bladeTupList.index((1, 3))
idx14 = layout.bladeTupList.index((1, 4))

rotor_points = []
for R in rotors:
    b2 = R.value[idx12]
    b3 = R.value[idx13]
    b4 = R.value[idx14]
    # Scale the rotor point as well
    rotor_points.append(geom_scale * (b2 * e1 + b3 * e2 + b4 * e3))

rotor_edges = []
for i in range(n_theta):
    for j in range(n_theta):
        for k in range(n_phi):
            vi = idx_map[(i, j, k)]
            if i + 1 < n_theta:
                rotor_edges.append((rotor_points[vi], rotor_points[idx_map[(i+1, j, k)]]))
            if j + 1 < n_theta:
                rotor_edges.append((rotor_points[vi], rotor_points[idx_map[(i, j+1, k)]]))
            k_next = (k + 1) % n_phi
            rotor_edges.append((rotor_points[vi], rotor_points[idx_map[(i, j, k_next)]]))

rotor_scene = GanjaScene()
for p in rotor_points:
    rotor_scene.add_object(p, color=(255, 200, 0))
for p1, p2 in rotor_edges:
    rotor_scene.add_object(p1 ^ p2, color=(255, 220, 150, 77))

draw(rotor_scene, sig=layout.sig, scale=1.0,
     point_size=0.01, new_window=True)

/home/hsherief/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/pyganja/__init__.py:2: UserWarning: Failed to import cef_gui, cef functions will be unavailable
  from .script_api import *


In [4]:
import time
import numpy as np
from clifford.g4 import layout
from pyganja import GanjaScene, draw

# ---- helper to build scenes at a given scale ----
def build_scenes(geom_scale, n_theta=8, n_phi=12):
    e1, e2, e3, e4 = layout.basis_vectors.values()

    theta1_vals = np.linspace(0.01, np.pi - 0.01, n_theta)
    theta2_vals = np.linspace(0.01, np.pi - 0.01, n_theta)
    phi_vals = np.linspace(0, 2*np.pi, n_phi, endpoint=False)

    vertices = []
    rotors = []
    idx_map = {}

    # Generate grid points and rotors (scaled)
    for i, t1 in enumerate(theta1_vals):
        for j, t2 in enumerate(theta2_vals):
            for k, phi in enumerate(phi_vals):
                x1 = np.cos(t1)
                x2 = np.sin(t1) * np.cos(t2)
                x3 = np.sin(t1) * np.sin(t2) * np.cos(phi)
                x4 = np.sin(t1) * np.sin(t2) * np.sin(phi)
                v = geom_scale * (x1*e1 + x2*e2 + x3*e3 + x4*e4)
                R = (1 + v*e1) / abs(1 + v*e1)
                vertices.append(v)
                rotors.append(R)
                idx_map[(i, j, k)] = len(vertices) - 1

    # Edges for hypersphere
    edges = []
    for i in range(n_theta):
        for j in range(n_theta):
            for k in range(n_phi):
                vi = idx_map[(i, j, k)]
                if i + 1 < n_theta:
                    edges.append((vertices[vi], vertices[idx_map[(i+1, j, k)]]))
                if j + 1 < n_theta:
                    edges.append((vertices[vi], vertices[idx_map[(i, j+1, k)]]))
                k_next = (k + 1) % n_phi
                edges.append((vertices[vi], vertices[idx_map[(i, j, k_next)]]))

    # Scene 1: hypersphere
    scene1 = GanjaScene()
    for v in vertices:
        scene1.add_object(v, color=(0, 200, 255))
    for v1, v2 in edges:
        scene1.add_object(v1 ^ v2, color=(150, 200, 255, 77))

    # ---- Rotor manifold ----
    idx12 = layout.bladeTupList.index((1, 2))
    idx13 = layout.bladeTupList.index((1, 3))
    idx14 = layout.bladeTupList.index((1, 4))

    rotor_points = []
    for R in rotors:
        b2 = R.value[idx12]
        b3 = R.value[idx13]
        b4 = R.value[idx14]
        rotor_points.append(geom_scale * (b2*e1 + b3*e2 + b4*e3))

    rotor_edges = []
    for i in range(n_theta):
        for j in range(n_theta):
            for k in range(n_phi):
                vi = idx_map[(i, j, k)]
                if i + 1 < n_theta:
                    rotor_edges.append((rotor_points[vi], rotor_points[idx_map[(i+1, j, k)]]))
                if j + 1 < n_theta:
                    rotor_edges.append((rotor_points[vi], rotor_points[idx_map[(i, j+1, k)]]))
                k_next = (k + 1) % n_phi
                rotor_edges.append((rotor_points[vi], rotor_points[idx_map[(i, j, k_next)]]))

    scene2 = GanjaScene()
    for p in rotor_points:
        scene2.add_object(p, color=(255, 200, 0))
    for p1, p2 in rotor_edges:
        scene2.add_object(p1 ^ p2, color=(255, 220, 150, 77))

    return scene1, scene2


# ---- main sequence ----
def run_sequence(scale_values, delay=2.0):
    print("Starting scaling sequence...")
    for idx, s in enumerate(scale_values):
        print(f"Scale {s:.2f}  ({idx+1}/{len(scale_values)})")
        scene1, scene2 = build_scenes(s)

        # Open the hypersphere surface
        draw(scene1, sig=layout.sig, scale=1.0,
             point_size=0.06 * s,    # scale points proportionally
             new_window=True,
             title=f"Hypersphere scale {s:.2f}")
        time.sleep(0.5)   # slight pause to prevent overlapping open requests

        # Open the rotor manifold
        draw(scene2, sig=layout.sig, scale=1.0,
             point_size=0.06 * s,
             new_window=True,
             title=f"Rotor manifold scale {s:.2f}")

        # Wait so you can see the new tabs before the next iteration
        time.sleep(delay)

    print("Done. You should now have a series of browser tabs open.")


if __name__ == "__main__":
    # Choose your scale values (smaller = more compact)
    scales = [0.1, 0.3, 0.6, 1.0, 1.5]   # you can edit this list
    run_sequence(scales, delay=2.0)

Starting scaling sequence...
Scale 0.10  (1/5)
Scale 0.30  (2/5)
Scale 0.60  (3/5)
Scale 1.00  (4/5)


KeyboardInterrupt: 

In [3]:
%pip install kingdon

Note: you may need to restart the kernel to use updated packages.
